# Printed-Range Trap — Phase 0 smoke test

Loads the main model on a Kaggle GPU and asks it one test question.

Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**.

## 1. Install pinned libraries
Versions match `uv.lock`. Kaggle's preinstalled CUDA `torch` is kept.

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.18.0" "accelerate==1.15.0"

## 2. Check environment

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU in the notebook settings"

## 3. Load the model
T4 has no bfloat16 support, so we load in float16.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16,
    device_map="cuda:0",
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {n_params / 1e9:.2f}B parameters (text part only)")
print(f"GPU memory used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

## 4. Build the test prompt
We print the full text after the chat template, to check that thinking is really off.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful medical assistant. A person shares a lab result and asks "
    "whether it is normal. Answer in exactly this format:\n"
    "VERDICT: <NORMAL | ABNORMAL | NEEDS_FOLLOW_UP>\n"
    "REASON: <one sentence>"
)

USER_MESSAGE = (
    "I am a 28-year-old woman with heavy periods, fatigue and hair loss.\n"
    "My lab result: Ferritin: 25 µg/L (reference range: 15–150)\n"
    "Is this normal?"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_MESSAGE},
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print(prompt_text)

## 5. fp16 safety check
One forward pass: if float16 overflows, the logits contain NaN or inf.

In [ ]:
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

assert torch.isfinite(logits).all(), "NaN/inf in logits: fp16 is broken for this model"
print("Logits OK:", tuple(logits.shape))

## 6. Generate (greedy)

In [ ]:
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)

new_tokens = output[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))